# x224 Benchmark 50k 5pct Dataset Build

This notebook prepares and validates the curated `224 x 224` WM-811K benchmark split used by the main anomaly-detection experiments.

It is meant to answer two questions clearly:

- can we regenerate the processed arrays and metadata from the raw pickle?
- do the outputs written under `data/processed/` match the intended split configuration?


## Notebook Flow

Run the notebook from top to bottom.

1. load the dataset config for this benchmark branch
2. confirm the raw pickle exists and inspect the raw label distribution
3. run the shared preparation script that writes arrays and metadata into `data/processed/x224/wm811k/`
4. validate the generated CSV and array files
5. inspect a few example wafers from the exported split


In [ ]:
try:
    import os
    import subprocess
    import sys
    from argparse import Namespace
    from pathlib import Path
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from IPython.display import display
    REPO_ROOT = Path.cwd().resolve()
    for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
        if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
            REPO_ROOT = candidate
            break
    else:
        raise FileNotFoundError('Could not locate the repository root.')
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    if str(REPO_ROOT / 'src') not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / 'src'))
    DATASET_DIR = REPO_ROOT / 'data' / 'dataset' / 'x64' / 'benchmark_50k_5pct'
    CONFIG_PATH = DATASET_DIR / 'data_config.toml'

    def repo_path(path_like: str | Path) -> Path:
        path = Path(path_like)
        return path if path.is_absolute() else REPO_ROOT / path
    (REPO_ROOT, CONFIG_PATH)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "import os\nimport subprocess\nimport sys\nfrom argparse import namespace\nfrom pathlib import path\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom ipython.display import display\nrepo_root = path.cwd().resolve()\nfor candidate in [repo_root, *repo_root.parents]:\n    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():\n        repo_root = candidate\n        break\nelse:\n    raise filenotfounderror('could not locate the repository root.')\nif str(repo_root) not in sys.path:\n    sys.path.insert(0, str(repo_root))\nif str(repo_root / 'src') not in sys.path:\n    sys.path.insert(0, str(repo_root / 'src'))\ndataset_dir = repo_root / 'data' / 'dataset' / 'x64' / 'benchmark_50k_5pct'\nconfig_path = dataset_dir / 'data_config.toml'\n\ndef repo_path(path_like: str | path) -> path:\n    path = path(path_like)\n    return path if path.is_absolute() else repo_root / path\n(repo_root, config_path)\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


In [ ]:
RUN_TRAINING = False
print(f'RUN_TRAINING = {RUN_TRAINING}')


In [ ]:
from wafer_defect.config import load_toml
from wafer_defect.data.legacy_pickle import read_legacy_pickle, unwrap_legacy_value
from scripts.prepare_wm811k import LABEL_DEFECT, LABEL_NORMAL, default_output_paths, infer_label_from_row
config = load_toml(CONFIG_PATH)
dataset_cfg = config['dataset']
split_cfg = config['splits']
dev_cfg = config['dev_subset']
train_subset_cfg = config.get('train_subset', {})
split_generation_cfg = config.get('split_generation', {})
labeled_split_cfg = config.get('labeled_split', {})
split_mode = str(split_generation_cfg.get('mode', 'normal_only_test_defects')).strip().lower()
cli_args = Namespace(config=str(CONFIG_PATH), dev=False, normal_limit=None, metadata_path=None)
metadata_path, arrays_dir = default_output_paths(Path(dataset_cfg['processed_root']), cli_args, dev_cfg, train_subset_cfg, split_mode=split_mode, labeled_split_cfg=labeled_split_cfg)
metadata_path = repo_path(metadata_path)
arrays_dir = repo_path(arrays_dir)
config_summary = pd.DataFrame([{'field': 'raw_pickle', 'value': dataset_cfg['raw_pickle']}, {'field': 'processed_root', 'value': dataset_cfg['processed_root']}, {'field': 'image_size', 'value': dataset_cfg['image_size']}, {'field': 'split_mode', 'value': split_mode}, {'field': 'train_normal_fraction', 'value': split_cfg['train_normal_fraction']}, {'field': 'val_normal_fraction', 'value': split_cfg['val_normal_fraction']}, {'field': 'test_normal_fraction', 'value': split_cfg['test_normal_fraction']}, {'field': 'train_subset.normal_count', 'value': train_subset_cfg.get('normal_count')}, {'field': 'train_subset.test_defect_fraction_of_test_normals', 'value': train_subset_cfg.get('test_defect_fraction_of_test_normals')}, {'field': 'expected_metadata_csv', 'value': metadata_path.relative_to(REPO_ROOT).as_posix()}, {'field': 'expected_arrays_dir', 'value': arrays_dir.relative_to(REPO_ROOT).as_posix()}])
display(config_summary)


## Raw Input Sanity Check

This cell checks that the raw pickle exists and that the label extraction logic sees a healthy mix of normal and defective wafers before we write any processed outputs.


In [ ]:
try:
    raw_pickle = repo_path(dataset_cfg['raw_pickle'])
    if not raw_pickle.exists():
        raise FileNotFoundError(f'Raw dataset file not found: {raw_pickle}')
    raw_df = read_legacy_pickle(raw_pickle).copy()
    raw_df['failureTypeText'] = raw_df['failureType'].map(unwrap_legacy_value)
    raw_df['label'] = raw_df.apply(infer_label_from_row, axis=1)
    usable_df = raw_df[raw_df['label'].notna()].copy()
    overview_df = pd.DataFrame([{'rows_in_pickle': len(raw_df), 'usable_rows': len(usable_df), 'normal_rows': int((usable_df['label'] == LABEL_NORMAL).sum()), 'defect_rows': int((usable_df['label'] == LABEL_DEFECT).sum())}])
    display(overview_df)
    display(usable_df['failureTypeText'].replace('', 'unlabeled').value_counts().head(10).rename_axis('failureTypeText').to_frame('count'))
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "raw_pickle = repo_path(dataset_cfg['raw_pickle'])\nif not raw_pickle.exists():\n    raise filenotfounderror(f'raw dataset file not found: {raw_pickle}')\nraw_df = read_legacy_pickle(raw_pickle).copy()\nraw_df['failuretypetext'] = raw_df['failuretype'].map(unwrap_legacy_value)\nraw_df['label'] = raw_df.apply(infer_label_from_row, axis=1)\nusable_df = raw_df[raw_df['label'].notna()].copy()\noverview_df = pd.dataframe([{'rows_in_pickle': len(raw_df), 'usable_rows': len(usable_df), 'normal_rows': int((usable_df['label'] == label_normal).sum()), 'defect_rows': int((usable_df['label'] == label_defect).sum())}])\ndisplay(overview_df)\ndisplay(usable_df['failuretypetext'].replace('', 'unlabeled').value_counts().head(10).rename_axis('failuretypetext').to_frame('count'))\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Generate Processed Outputs

This cell calls the shared preparation script used by the rest of the repository. It writes the metadata CSV and the `.npy` wafer arrays under `data/processed/x224/wm811k/` according to the config in this folder.


In [ ]:
try:
    command = [sys.executable, 'scripts/prepare_wm811k.py', '--config', str(CONFIG_PATH.relative_to(REPO_ROOT))]
    pythonpath_entries = [str(REPO_ROOT), str(REPO_ROOT / 'src')]
    existing_pythonpath = os.environ.get('PYTHONPATH')
    if existing_pythonpath:
        pythonpath_entries.append(existing_pythonpath)
    run_env = os.environ.copy()
    run_env['PYTHONPATH'] = os.pathsep.join(pythonpath_entries)
    result = subprocess.run(command, cwd=REPO_ROOT, check=True, capture_output=True, text=True, env=run_env)
    print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "command = [sys.executable, 'scripts/prepare_wm811k.py', '--config', str(config_path.relative_to(repo_root))]\npythonpath_entries = [str(repo_root), str(repo_root / 'src')]\nexisting_pythonpath = os.environ.get('pythonpath')\nif existing_pythonpath:\n    pythonpath_entries.append(existing_pythonpath)\nrun_env = os.environ.copy()\nrun_env['pythonpath'] = os.pathsep.join(pythonpath_entries)\nresult = subprocess.run(command, cwd=repo_root, check=true, capture_output=true, text=true, env=run_env)\nprint(result.stdout)\nif result.stderr.strip():\n    print(result.stderr)\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Validate the Written Metadata and Arrays

We now confirm that the outputs exist, that split counts look sensible, and that the array folder agrees with the CSV.


In [ ]:
try:
    if not metadata_path.exists():
        raise FileNotFoundError(f'Expected metadata file was not created: {metadata_path}')
    if not arrays_dir.exists():
        raise FileNotFoundError(f'Expected arrays directory was not created: {arrays_dir}')
    metadata = pd.read_csv(metadata_path)
    display(metadata.head())
    display(metadata.groupby(['split', 'is_anomaly']).size().rename('count').reset_index().sort_values(['split', 'is_anomaly']))
    array_files = sorted(arrays_dir.glob('*.npy'))
    array_file_names = {path.name for path in array_files}
    metadata_file_names = set(metadata['array_path'].map(lambda value: Path(value).name))
    missing_arrays = sorted(metadata_file_names - array_file_names)
    extra_arrays = sorted(array_file_names - metadata_file_names)
    validation_df = pd.DataFrame([{'metadata_rows': len(metadata), 'array_files': len(array_files), 'missing_arrays': len(missing_arrays), 'extra_arrays': len(extra_arrays)}])
    display(validation_df)
    assert len(missing_arrays) == 0, f'Metadata references missing arrays, e.g. {missing_arrays[:5]}'
    assert len(extra_arrays) == 0, f'Arrays directory contains unexpected extra files, e.g. {extra_arrays[:5]}'
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "if not metadata_path.exists():\n    raise filenotfounderror(f'expected metadata file was not created: {metadata_path}')\nif not arrays_dir.exists():\n    raise filenotfounderror(f'expected arrays directory was not created: {arrays_dir}')\nmetadata = pd.read_csv(metadata_path)\ndisplay(metadata.head())\ndisplay(metadata.groupby(['split', 'is_anomaly']).size().rename('count').reset_index().sort_values(['split', 'is_anomaly']))\narray_files = sorted(arrays_dir.glob('*.npy'))\narray_file_names = {path.name for path in array_files}\nmetadata_file_names = set(metadata['array_path'].map(lambda value: path(value).name))\nmissing_arrays = sorted(metadata_file_names - array_file_names)\nextra_arrays = sorted(array_file_names - metadata_file_names)\nvalidation_df = pd.dataframe([{'metadata_rows': len(metadata), 'array_files': len(array_files), 'missing_arrays': len(missing_arrays), 'extra_arrays': len(extra_arrays)}])\ndisplay(validation_df)\nassert len(missing_arrays) == 0, f'metadata references missing arrays, e.g. {missing_arrays[:5]}'\nassert len(extra_arrays) == 0, f'arrays directory contains unexpected extra files, e.g. {extra_arrays[:5]}'\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


In [ ]:
try:
    sample_rows = metadata.sample(n=min(6, len(metadata)), random_state=int(split_cfg['random_seed']))
    sample_shapes = []
    for _, row in sample_rows.iterrows():
        wafer = np.load(repo_path(row['array_path']))
        sample_shapes.append({'array_path': row['array_path'], 'shape': tuple(wafer.shape), 'min': float(wafer.min()), 'max': float(wafer.max())})
    display(pd.DataFrame(sample_shapes))
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "sample_rows = metadata.sample(n=min(6, len(metadata)), random_state=int(split_cfg['random_seed']))\nsample_shapes = []\nfor _, row in sample_rows.iterrows():\n    wafer = np.load(repo_path(row['array_path']))\n    sample_shapes.append({'array_path': row['array_path'], 'shape': tuple(wafer.shape), 'min': float(wafer.min()), 'max': float(wafer.max())})\ndisplay(pd.dataframe(sample_shapes))\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metadata['split'].value_counts().plot(kind='bar', ax=axes[0], title='Split Distribution')
metadata['is_anomaly'].map({0: 'normal', 1: 'anomaly'}).value_counts().plot(kind='bar', ax=axes[1], title='Class Distribution')
metadata['defect_type'].value_counts().head(8).plot(kind='bar', ax=axes[2], title='Top Defect Types')
for ax in axes:
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()


In [ ]:
try:

    def load_wafer(relative_path: str) -> np.ndarray:
        return np.load(repo_path(relative_path))
    normal_examples = metadata[metadata['is_anomaly'] == 0].sample(6, random_state=42)
    anomaly_examples = metadata[metadata['is_anomaly'] == 1].sample(6, random_state=42)
    fig, axes = plt.subplots(2, 6, figsize=(14, 5))
    for idx, (_, row) in enumerate(normal_examples.iterrows()):
        axes[0, idx].imshow(load_wafer(row['array_path']), cmap='viridis')
        axes[0, idx].set_title(f"Normal\n{row['split']}")
        axes[0, idx].axis('off')
    for idx, (_, row) in enumerate(anomaly_examples.iterrows()):
        axes[1, idx].imshow(load_wafer(row['array_path']), cmap='viridis')
        axes[1, idx].set_title(f"{row['defect_type']}")
        axes[1, idx].axis('off')
    plt.tight_layout()
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = 'def load_wafer(relative_path: str) -> np.ndarray:\n    return np.load(repo_path(relative_path))\nnormal_examples = metadata[metadata[\'is_anomaly\'] == 0].sample(6, random_state=42)\nanomaly_examples = metadata[metadata[\'is_anomaly\'] == 1].sample(6, random_state=42)\nfig, axes = plt.subplots(2, 6, figsize=(14, 5))\nfor idx, (_, row) in enumerate(normal_examples.iterrows()):\n    axes[0, idx].imshow(load_wafer(row[\'array_path\']), cmap=\'viridis\')\n    axes[0, idx].set_title(f"normal\\n{row[\'split\']}")\n    axes[0, idx].axis(\'off\')\nfor idx, (_, row) in enumerate(anomaly_examples.iterrows()):\n    axes[1, idx].imshow(load_wafer(row[\'array_path\']), cmap=\'viridis\')\n    axes[1, idx].set_title(f"{row[\'defect_type\']}")\n    axes[1, idx].axis(\'off\')\nplt.tight_layout()\n'
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Adapting This Pattern

When we curate more dataset branches such as `x224/benchmark_50k_5pct`, `x240/benchmark_50k_5pct`, or `x64/holdout70k_3p5k`, they should follow the same notebook structure:

- local `data_config.toml`
- one generation cell that calls the shared prep script
- one validation block that confirms the CSV and arrays were written correctly
